# Rede neural com uma camada oculta

Construímos uma rede de alimentação direta com uma camada oculta e uma saída para classificar controle (0) ou diabetes (1). Usamos sigmoide, erro quadrático e retropropagação, conceitos apresentados na aula 6.

A rede está implementada do zero com NumPy para as contas. A arquitetura continua pequena, com 2, 4, 8 ou 12 neurônios ocultos. As saídas ficam salvas para leitura. Para reexecutar, o kernel precisa ter NumPy.

A submissão é preparada ao final, usando a configuração escolhida na validação cruzada e todas as 520 amostras rotuladas como referência de aprendizado.

### Referências das aulas e decisões da implementação

| Parte | Referência, página do PDF |
|---|---|
| Soma ponderada e entrada fixa para o viés | Aula 6, página 6 |
| Sigmoide | Aula 6, página 7 |
| Alimentação direta e camada oculta | Aula 6, páginas 9, 10 e 12 |
| Saída binária com limite 0,5 | Aula 6, página 11 |
| Erro quadrático e descida do gradiente | Aula 6, páginas 15 e 16 |
| Épocas de treinamento | Aula 6, página 17 |
| Escolha da quantidade de neurônios por validação cruzada | Aula 6, página 19 |
| Retropropagação do erro | Aula 6, páginas 21 a 23 |
| Discretização, binarização e seleção de atributos | Aula 2 |
| Normalização pelo mínimo e máximo | Aula 5, página 12 |
| Normalização pela média e desvio padrão | Aula 10, página 16 |
| Divisões e métricas | Aula 7 |

O slide 17 descreve atualizações exemplo a exemplo. Nesta implementação, calculamos a média dos gradientes de todos os exemplos do treinamento e fazemos uma atualização por época. Essa atualização em lote é uma escolha de processamento desta aplicação. Ela usa as mesmas derivadas da sigmoide e do erro quadrático.

Também escolhemos os grupos de atributos, a inicialização aleatória, as taxas e a quantidade de épocas comparadas. As páginas são contadas a partir da capa dos PDFs.

### Resultado desta execução

Escolhemos a configuração pelo F1 macro médio nas cinco partes de treinamento. A rede usa **4 entradas, 12 neurônios ocultos e uma saída**, com 73 pesos ajustáveis, incluindo os vieses.

| Escolha | Configuração |
|---|---|
| Atributos | SYSTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES |
| Normalização | média e desvio padrão |
| Preenchimento dos vazios numéricos | mediana calculada no treinamento |
| Taxa de aprendizado | 4.0 |
| Épocas | 4000 |
| Inicialização usada no modelo final | Semente 7 |
| Ativação | Sigmoide na camada oculta e na saída |
| Aprendizado | Retropropagação do erro quadrático, atualização em lote |
| Decisão | Saída maior que 0,5 indica classe 1 |

| Avaliação | Acurácia | F1 macro |
|---|---|---|
| Escolha: divisão 42, pesos 7 | 88.94% | 88.93% |
| Conferência: divisão 73, pesos 7 | 89.91% | 89.90% |
| Conferência: divisão 101, pesos 7 | 88.47% | 88.46% |
| Conferência: divisão 42, pesos 19 | 87.02% | 87.00% |
| Conferência: divisão 42, pesos 31 | 87.50% | 87.49% |
| 104 amostras reservadas | 83.65% | 83.65% |

Comparamos 576 configurações iniciais e mais oito extensões de treinamento das quatro trajetórias mais bem avaliadas. As conferências mantêm a configuração escolhida. Elas mostram a variação causada pelas divisões dos dados e pelos pesos iniciais. Não são conjuntos externos novos.

Na reserva, acertamos 87 de 104 amostras. O F1 macro da reserva é 83.65%. Essa avaliação já foi consultada no projeto e não garante o resultado do Kaggle.

O CSV para envio é `resultados/submissao_rede_neural_aulas.csv`, com 312 previsões. Ele é produzido por uma nova rede treinada com todas as 520 amostras rotuladas, usando a mesma configuração. O envio ao Kaggle ainda precisa ser feito.

### Base e divisões

Repetimos a divisão dos modelos anteriores. Temos 416 amostras para escolher a configuração e 104 reservadas para a comparação local. Cada paciente aparece uma única vez na base padronizada.

Escolhemos a rede com cinco partes da semente 42. A semente de inicialização dos pesos é 7 e fica fixa em toda a busca. Depois conferimos a configuração escolhida em outras divisões e com outras inicializações, sem escolher uma semente apenas pelo melhor resultado. As 104 amostras já foram consultadas no projeto.

In [1]:
from pathlib import Path
from itertools import product
from time import perf_counter
import csv
import hashlib
import json
import random
import sys

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

numericas = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE",
    "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
categoricas = ("SMOKER", "FAMILY_HISTORY", "PHYSICAL_ACTIVITY", "GENDER", "BLOOD_GROUP", "RH_FACTOR")
grupos = {
    "Três exames": ("GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
    "Quatro medidas": ("SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"),
    "Nove medidas": numericas,
    "Medidas e categorias": numericas + categoricas,
}
classes = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os registros por paciente.")
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos as classes 0 e 1.")

controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, semente):
    partes = [[] for _ in range(5)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % 5].append(i)
    for posicao, parte in enumerate(partes):
        referencia = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(referencia), np.array(parte)


print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")
print(f"Treinamento: {len(treinamento)} | Reservadas: {len(reservadas)}")
print("SHA-256:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())

Python 3.12.14 | NumPy 2.3.5
Treinamento: 416 | Reservadas: 104
SHA-256: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032


### Preparação das entradas

Comparamos mínimo e máximo com média e desvio padrão. Os parâmetros são calculados só na referência de cada rodada. Também comparamos preencher ausências pela média ou mediana da referência.

Mantemos a leitura numérica anterior, incluindo medidas não positivas como ausentes e HEMOGLOBIN acima de 100 como ausente sob a premissa anterior de HbA1c. As categorias, quando usadas, viram colunas de 0 ou 1. A referência define as categorias conhecidas. Reservamos uma categoria para desconhecido e outra para um valor inédito.

A função de transformação recebe os parâmetros prontos. Assim, a avaliação e o teste usam os mesmos valores que foram calculados durante o treinamento.

#### Como preparamos uma amostra

Começamos separando medidas numéricas e categorias. Mantemos as medidas numéricas contínuas. Os valores ausentes precisam receber uma estimativa para que a rede consiga fazer as contas; comparamos média e mediana. Calculamos essa estimativa apenas nos exemplos usados para treinar em cada divisão.

Também comparamos duas normalizações. Pelo mínimo e máximo, subtraímos o menor valor e dividimos pela amplitude. Pela média e desvio padrão, subtraímos a média e dividimos pelo desvio. Aplicamos às amostras de avaliação os parâmetros encontrados no treinamento.

No grupo que inclui categorias, criamos uma coluna binária para cada categoria. Um valor desconhecido tem sua própria categoria. Se aparecer na avaliação uma categoria que não estava no treinamento, usamos uma coluna reservada para esse caso. A lista de categorias também é construída somente no treinamento.

Mantemos a premissa anterior de que `HEMOGLOBIN` representa HbA1c. Valores maiores que 100 nesse campo são tratados como ausentes nesta aplicação. Essa é uma decisão sobre a base, e não uma regra aprendida pela rede.

In [2]:
def ler_numero(amostra, coluna):
    try:
        valor = float(str(amostra[coluna]).strip().replace(",", "."))
    except ValueError:
        return np.nan
    if not np.isfinite(valor) or valor <= 0:
        return np.nan
    if coluna == "HEMOGLOBIN" and valor > 100:
        return np.nan
    return valor


def ler_categoria(amostra, coluna):
    valor = amostra[coluna].strip()
    if valor in {"", "desconhecido"} or (coluna == "GENDER" and valor == "N"):
        return "__desconhecido__"
    return valor


def ajustar_preparo(referencia, grupo, escala, preenchimento):
    parametros = {"grupo": grupo, "escala": escala, "preenchimento": preenchimento, "numeros": {}, "categorias": {}}
    entradas = []
    for coluna in grupos[grupo]:
        if coluna in categoricas:
            valores = {ler_categoria(a, coluna) for a in referencia}
            valores.update(("__desconhecido__", "__nao_visto__"))
            parametros["categorias"][coluna] = sorted(valores)
            entradas.extend(f"{coluna}__{valor}" for valor in sorted(valores))
            continue

        valores = np.array([ler_numero(a, coluna) for a in referencia])
        disponiveis = valores[np.isfinite(valores)]
        if not len(disponiveis):
            raise ValueError(f"Nenhuma medida disponível para {coluna}.")
        estimativa = np.mean(disponiveis) if preenchimento == "media" else np.median(disponiveis)
        if escala == "minmax":
            centro = np.min(disponiveis)
            divisor = np.ptp(disponiveis)
        else:
            centro = np.mean(disponiveis)
            divisor = np.std(disponiveis, ddof=1) if len(disponiveis) > 1 else 0
        parametros["numeros"][coluna] = {
            "estimativa": float(estimativa), "centro": float(centro), "divisor": float(divisor),
        }
        entradas.append(coluna)
    parametros["entradas"] = entradas
    return parametros


def transformar(consulta, parametros):
    linhas = []
    for a in consulta:
        linha = []
        for coluna in grupos[parametros["grupo"]]:
            if coluna in parametros["categorias"]:
                valores = parametros["categorias"][coluna]
                categoria = ler_categoria(a, coluna)
                if categoria not in valores:
                    categoria = "__nao_visto__"
                linha.extend(float(categoria == valor) for valor in valores)
            else:
                p = parametros["numeros"][coluna]
                valor = ler_numero(a, coluna)
                if not np.isfinite(valor):
                    valor = p["estimativa"]
                linha.append((valor - p["centro"]) / p["divisor"] if p["divisor"] else 0.0)
        linhas.append(linha)
    return np.array(linhas, dtype=float)

### Passagem das entradas até a saída

Cada neurônio recebe uma soma das entradas multiplicadas por seus pesos e aplica a sigmoide. A sigmoide transforma essa soma em um número entre 0 e 1.

Primeiro calculamos a camada oculta. Depois suas saídas alimentam o neurônio de saída. Acrescentamos uma entrada fixa de -1, como no slide 6, para representar o viés. A previsão é diabetes se a saída for maior que 0,5. Em igualdade exata, usamos controle.

Os pesos começam em valores aleatórios entre -0,5 e 0,5. A semente fixa permite repetir essa inicialização.

#### O caminho de uma previsão

Uma amostra entra como uma lista de valores já preparados. Cada neurônio da camada oculta multiplica esses valores por seus pesos, soma os resultados e aplica a sigmoide. Os neurônios podem aprender combinações diferentes das mesmas entradas.

O neurônio de saída recebe os resultados da camada oculta e repete essa conta. Sua resposta fica entre 0 e 1. Usamos o limite 0,5: acima disso, prevemos diabetes; caso contrário, controle. Esse número não foi calibrado como uma probabilidade clínica.

O viés permite deslocar a resposta do neurônio. Nas contas, ele aparece como uma entrada fixa de valor -1, seguindo a convenção do material da aula.

In [3]:
def sigmoide(z):
    resultado = np.empty_like(z, dtype=float)
    positivos = z >= 0
    resultado[positivos] = 1 / (1 + np.exp(-z[positivos]))
    exponencial = np.exp(z[~positivos])
    resultado[~positivos] = exponencial / (1 + exponencial)
    return resultado


def incluir_vies(x):
    return np.column_stack((x, -np.ones(len(x))))


def passar(x_com_vies, pesos_entrada, pesos_saida):
    oculta = sigmoide(x_com_vies @ pesos_entrada)
    oculta_com_vies = incluir_vies(oculta)
    saida = sigmoide(oculta_com_vies @ pesos_saida)
    return oculta, oculta_com_vies, saida


def prever(x, modelo):
    _, _, saida = passar(incluir_vies(x), modelo[0], modelo[1])
    return (saida.ravel() > 0.5).astype(np.int8)

### Aprendizado por retropropagação

Medimos o erro como metade da média dos quadrados das diferenças entre as saídas e as classes reais. A cada época, calculamos as saídas e o erro, propagamos o erro da saída para a camada oculta e ajustamos os pesos.

Na saída, multiplicamos o erro pela derivada da sigmoide, saída × (1 − saída). Na camada oculta, o erro recebido da saída é multiplicado pelos pesos e pela derivada das ativações ocultas. A taxa de aprendizagem determina o tamanho de cada ajuste.

As fórmulas de atualização e retropropagação estão nas páginas 16 e 21 da aula 6. Aqui somamos as contribuições dos exemplos e dividimos pela quantidade de amostras, fazendo uma atualização em lote por época.

#### Como os pesos mudam

Os pesos começam com valores aleatórios pequenos. A semente permite repetir essa mesma inicialização. Durante o treinamento, comparamos a saída da rede com a classe conhecida e calculamos o erro quadrático.

A retropropagação calcula quanto cada peso contribuiu para esse erro. A taxa de aprendizado controla o tamanho do ajuste. Uma época corresponde a uma passagem pelos exemplos de treinamento. Aqui acumulamos os gradientes dos exemplos e usamos a média para fazer uma atualização por época, conforme a diferença de processamento registrada no início.

Repetimos esse processo e guardamos a rede nas quantidades de épocas que queremos comparar. Ter menor erro nos exemplos de treinamento não garante melhor classificação de exemplos novos; por isso usamos a validação cruzada para escolher a configuração.

In [4]:
def treinar(x, y, ocultos, taxa, pontos, semente=7, registrar=False):
    gerador = np.random.default_rng(semente)
    pesos_entrada = gerador.uniform(-0.5, 0.5, size=(x.shape[1] + 1, ocultos))
    pesos_saida = gerador.uniform(-0.5, 0.5, size=(ocultos + 1, 1))
    x_com_vies = incluir_vies(x)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    modelos = {}
    historico = []

    for epoca in range(1, max(pontos) + 1):
        oculta, oculta_com_vies, saida = passar(x_com_vies, pesos_entrada, pesos_saida)
        delta_saida = (y - saida) * saida * (1 - saida)
        delta_oculta = (delta_saida @ pesos_saida[:-1].T) * oculta * (1 - oculta)
        ajuste_saida = oculta_com_vies.T @ delta_saida / len(x)
        ajuste_entrada = x_com_vies.T @ delta_oculta / len(x)
        pesos_saida += taxa * ajuste_saida
        pesos_entrada += taxa * ajuste_entrada

        if epoca in pontos or (registrar and epoca % 100 == 0):
            _, _, nova_saida = passar(x_com_vies, pesos_entrada, pesos_saida)
            erro = float(np.mean((y - nova_saida) ** 2) / 2)
            if not np.isfinite(erro):
                raise ValueError("O treinamento gerou um erro não finito.")
            if registrar:
                historico.append((epoca, erro))
            if epoca in pontos:
                modelos[epoca] = (pesos_entrada.copy(), pesos_saida.copy())
    return modelos, historico


def contar(reais, previstas):
    return np.array([
        np.sum((reais == 0) & (previstas == 0)),
        np.sum((reais == 0) & (previstas == 1)),
        np.sum((reais == 1) & (previstas == 0)),
        np.sum((reais == 1) & (previstas == 1)),
    ], dtype=np.int16)


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(contagens, -1, 0)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (razao(2 * tn, 2 * tn + fp + fn) + razao(2 * tp, 2 * tp + fp + fn)) / 2,
        "precisao": (razao(tn, tn + fn) + razao(tp, tp + fp)) / 2,
        "precisao0": razao(tn, tn + fn), "precisao1": razao(tp, tp + fp),
        "recall0": razao(tn, tn + fp), "recall1": razao(tp, tp + fn),
    }


def mostrar(nome, contagens):
    m = metricas(contagens)
    print(f"{nome:25} | " + " | ".join(
        f"{np.mean(m[c]):>8.2%}" for c in ("acuracia", "f1", "precisao", "recall0", "recall1")
    ))

### Escolha da configuração

Comparamos quatro grupos de atributos, duas normalizações, média e mediana para ausências, quatro tamanhos de camada oculta e três taxas de aprendizagem. Lemos as métricas após 500, 1.500 e 4.000 épocas.

Escolhemos pelo F1 macro médio nas cinco rodadas com semente 42. Em empate, preferimos menos atributos de origem, menos neurônios ocultos, menos épocas e menor taxa. A ordem fixa completa os demais empates.

Os quatro melhores trajetos de treinamento dessa etapa são estendidos para comparar 8.000 e 16.000 épocas. Essa extensão usa as mesmas divisões de escolha. As conferências e a reserva vêm depois da escolha final.

In [5]:
escalas = ("minmax", "desvio")
preenchimentos = ("mediana", "media")
ocultos_testados = (2, 4, 8, 12)
taxas = (0.25, 1.0, 4.0)
pontos_iniciais = (500, 1500, 4000)
trajetos = list(product(grupos, escalas, preenchimentos, ocultos_testados, taxas))
configuracoes = [trajeto + (epoca,) for trajeto in trajetos for epoca in pontos_iniciais]


def dados_da_divisao(indices_ref, indices_aval, grupo, escala, preenchimento):
    ref = [amostras[i] for i in indices_ref]
    aval = [amostras[i] for i in indices_aval]
    parametros = ajustar_preparo(ref, grupo, escala, preenchimento)
    return transformar(ref, parametros), transformar(aval, parametros), parametros


inicio = perf_counter()
rodadas = []
for numero, (indices_ref, indices_aval) in enumerate(dividir(treinamento, 42), start=1):
    cache = {}
    resultados_rodada = []
    for grupo, escala, preenchimento, ocultos, taxa in trajetos:
        chave = (grupo, escala, preenchimento)
        if chave not in cache:
            cache[chave] = dados_da_divisao(indices_ref, indices_aval, *chave)
        x, aval, _ = cache[chave]
        modelos, _ = treinar(x, classes[indices_ref], ocultos, taxa, pontos_iniciais)
        for epoca in pontos_iniciais:
            resultados_rodada.append(contar(classes[indices_aval], prever(aval, modelos[epoca])))
    rodadas.append(resultados_rodada)
    print(f"Rodada {numero} concluída", flush=True)

rodadas = np.array(rodadas)
f1_inicial = metricas(rodadas)["f1"].mean(axis=0)


def chave_config(i, medias_f1, grade):
    grupo, escala, preenchimento, ocultos, taxa, epoca = grade[i]
    return (-medias_f1[i], len(grupos[grupo]), ocultos, epoca, taxa, i)


ordem_inicial = sorted(range(len(configuracoes)), key=lambda i: chave_config(i, f1_inicial, configuracoes))
finalistas = []
for i in ordem_inicial:
    trajeto = configuracoes[i][:-1]
    if trajeto not in finalistas:
        finalistas.append(trajeto)
    if len(finalistas) == 4:
        break

print(f"Configurações iniciais: {len(configuracoes)}")
print(f"Tempo da busca inicial: {perf_counter() - inicio:.1f} s")
for i in ordem_inicial[:10]:
    print(f"F1 {f1_inicial[i]:.2%} | {configuracoes[i]}")

Rodada 1 concluída
Rodada 2 concluída
Rodada 3 concluída
Rodada 4 concluída
Rodada 5 concluída
Configurações iniciais: 576
Tempo da busca inicial: 365.3 s
F1 88.93% | ('Quatro medidas', 'desvio', 'mediana', 12, 4.0, 4000)
F1 87.73% | ('Três exames', 'desvio', 'mediana', 8, 4.0, 4000)
F1 87.47% | ('Quatro medidas', 'desvio', 'mediana', 8, 4.0, 4000)
F1 87.26% | ('Três exames', 'desvio', 'mediana', 12, 4.0, 4000)
F1 87.02% | ('Três exames', 'desvio', 'mediana', 4, 4.0, 4000)
F1 85.78% | ('Quatro medidas', 'desvio', 'media', 12, 4.0, 4000)
F1 84.31% | ('Nove medidas', 'desvio', 'mediana', 12, 4.0, 4000)
F1 82.46% | ('Três exames', 'desvio', 'media', 8, 4.0, 4000)
F1 81.93% | ('Quatro medidas', 'desvio', 'mediana', 8, 4.0, 1500)
F1 81.49% | ('Quatro medidas', 'desvio', 'mediana', 4, 4.0, 4000)


#### Por que estendemos apenas algumas configurações

A primeira comparação usa até 4.000 épocas. Depois repetimos o treinamento das quatro trajetórias mais bem avaliadas e observamos os resultados com 8.000 e 16.000 épocas. Cada trajetória mantém o mesmo grupo de atributos, normalização, preenchimento, número de neurônios e taxa.

Isso nos permite verificar se mais tempo de aprendizado ajuda essas candidatas. A busca é finita e usa a mesma divisão para escolher as configurações. Por isso, conferimos a escolha em outras divisões em seguida.

In [6]:
pontos_extensao = (8000, 16000)
configuracoes_extensao = [trajeto + (epoca,) for trajeto in finalistas for epoca in pontos_extensao]
rodadas_extensao = []

for numero, (indices_ref, indices_aval) in enumerate(dividir(treinamento, 42), start=1):
    resultados_rodada = []
    for grupo, escala, preenchimento, ocultos, taxa in finalistas:
        x, aval, _ = dados_da_divisao(indices_ref, indices_aval, grupo, escala, preenchimento)
        modelos, _ = treinar(x, classes[indices_ref], ocultos, taxa, pontos_extensao)
        for epoca in pontos_extensao:
            resultados_rodada.append(contar(classes[indices_aval], prever(aval, modelos[epoca])))
    rodadas_extensao.append(resultados_rodada)
    print(f"Extensão, rodada {numero} concluída", flush=True)

configuracoes_completas = configuracoes + configuracoes_extensao
contagens_completas = np.concatenate((rodadas, np.array(rodadas_extensao)), axis=1)
f1_completo = metricas(contagens_completas)["f1"].mean(axis=0)
ordem_completa = sorted(
    range(len(configuracoes_completas)),
    key=lambda i: chave_config(i, f1_completo, configuracoes_completas)
)
escolhida = ordem_completa[0]
configuracao_final = configuracoes_completas[escolhida]

print("Melhores configurações após a extensão")
for i in ordem_completa[:10]:
    acc = metricas(contagens_completas[:, i])["acuracia"].mean()
    print(f"Acc {acc:.2%} | F1 {f1_completo[i]:.2%} | {configuracoes_completas[i]}")
print("Escolhida:", configuracao_final)

Extensão, rodada 1 concluída
Extensão, rodada 2 concluída
Extensão, rodada 3 concluída
Extensão, rodada 4 concluída
Extensão, rodada 5 concluída
Melhores configurações após a extensão
Acc 88.94% | F1 88.93% | ('Quatro medidas', 'desvio', 'mediana', 12, 4.0, 4000)
Acc 88.70% | F1 88.69% | ('Três exames', 'desvio', 'mediana', 12, 4.0, 8000)
Acc 87.98% | F1 87.96% | ('Três exames', 'desvio', 'mediana', 8, 4.0, 8000)
Acc 87.97% | F1 87.94% | ('Quatro medidas', 'desvio', 'mediana', 12, 4.0, 16000)
Acc 87.73% | F1 87.73% | ('Três exames', 'desvio', 'mediana', 8, 4.0, 4000)
Acc 87.50% | F1 87.47% | ('Quatro medidas', 'desvio', 'mediana', 8, 4.0, 4000)
Acc 87.49% | F1 87.47% | ('Três exames', 'desvio', 'mediana', 8, 4.0, 16000)
Acc 87.50% | F1 87.47% | ('Quatro medidas', 'desvio', 'mediana', 12, 4.0, 8000)
Acc 87.26% | F1 87.26% | ('Três exames', 'desvio', 'mediana', 12, 4.0, 4000)
Acc 87.25% | F1 87.23% | ('Três exames', 'desvio', 'mediana', 12, 4.0, 16000)
Escolhida: ('Quatro medidas', 'desv

### Conferências e comparação local

Fixamos grupo, escala, preenchimento, quantidade de neurônios, taxa e épocas. Repetimos a rede com as sementes de divisão 73 e 101, mantendo a inicialização 7. Depois mantemos a divisão 42 e usamos inicializações 19 e 31 para acompanhar o efeito dos pesos iniciais. Essas conferências não escolhem uma nova configuração.

Por fim, treinamos nas 416 amostras e avaliamos nas mesmas 104 reservadas. A tabela inclui as métricas já registradas dos outros modelos nessa divisão.

In [7]:
def ajustar_final(indices_ref, indices_aval, semente_pesos=7, registrar=False):
    grupo, escala, preenchimento, ocultos, taxa, epoca = configuracao_final
    x, aval, parametros = dados_da_divisao(indices_ref, indices_aval, grupo, escala, preenchimento)
    modelos, historico = treinar(
        x, classes[indices_ref], ocultos, taxa, (epoca,), semente_pesos, registrar
    )
    modelo = modelos[epoca]
    return contar(classes[indices_aval], prever(aval, modelo)), modelo, parametros, historico


conferencias = {}
for semente_divisao, semente_pesos in ((73, 7), (101, 7), (42, 19), (42, 31)):
    contagens = []
    for ref, aval in dividir(treinamento, semente_divisao):
        contagens.append(ajustar_final(ref, aval, semente_pesos)[0])
    conferencias[semente_divisao, semente_pesos] = np.array(contagens)

print("Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
mostrar("Escolha, divisão 42", contagens_completas[:, escolhida])
for (semente_divisao, semente_pesos), contagens in conferencias.items():
    mostrar(f"Divisão {semente_divisao}, pesos {semente_pesos}", contagens)

reserva, modelo_reserva, preparo_reserva, historico_reserva = ajustar_final(
    treinamento, reservadas, registrar=True
)
print()
print("Nas 104 amostras reservadas")
mostrar("Primeiro KNN enviado", np.array([44, 6, 8, 46]))
mostrar("Árvore quatro atributos", np.array([43, 7, 9, 45]))
mostrar("Mistura KNN e árvore", np.array([44, 6, 7, 47]))
mostrar("Rede neural", reserva)
print("Matriz da rede:", reserva.reshape(2, 2).tolist())
print("Acertos:", int(reserva[0] + reserva[3]), "de", len(reservadas))
print("Precisão controle:", f"{metricas(reserva)['precisao0']:.2%}")
print("Precisão diabetes:", f"{metricas(reserva)['precisao1']:.2%}")

Avaliação                 | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Escolha, divisão 42       |   88.94% |   88.93% |   89.02% |   90.00% |   87.94%
Divisão 73, pesos 7       |   89.91% |   89.90% |   90.17% |   91.50% |   88.46%
Divisão 101, pesos 7      |   88.47% |   88.46% |   88.56% |   89.00% |   87.99%
Divisão 42, pesos 19      |   87.02% |   87.00% |   87.02% |   87.00% |   87.04%
Divisão 42, pesos 31      |   87.50% |   87.49% |   87.50% |   88.00% |   87.04%

Nas 104 amostras reservadas
Primeiro KNN enviado      |   86.54% |   86.53% |   86.54% |   88.00% |   85.19%
Árvore quatro atributos   |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Mistura KNN e árvore      |   87.50% |   87.49% |   87.48% |   88.00% |   87.04%
Rede neural               |   83.65% |   83.65% |   83.70% |   86.00% |   81.48%
Matriz da rede: [[43, 7], [10, 44]]
Acertos: 87 de 104
Precisão controle: 81.13%
Precisão diabetes: 86.27%


### Arquitetura escolhida e evolução do aprendizado

Cada entrada corresponde a uma medida normalizada ou a uma coluna binária de categoria. O viés é uma entrada fixa adicional nas contas. A camada oculta tem a quantidade escolhida na validação, e a saída contém um único neurônio sigmoide.

A tabela de erro mostra como o erro quadrático de treinamento evoluiu. Esse erro serve para acompanhar o aprendizado e é diferente das métricas de classificação da validação.

In [8]:
ocultos_finais = configuracao_final[3]
numero_entradas = len(preparo_reserva["entradas"])
numero_pesos = modelo_reserva[0].size + modelo_reserva[1].size
print(f"Arquitetura: {numero_entradas} entradas -> {ocultos_finais} neurônios ocultos -> 1 saída")
print("Pesos ajustáveis, incluindo os vieses:", numero_pesos)
print("Entradas:", ", ".join(preparo_reserva["entradas"]))
print("Configuração:", configuracao_final)
print("Época | Erro quadrático de treinamento")
for epoca, erro in historico_reserva[::max(1, len(historico_reserva) // 10)]:
    print(f"{epoca:>5} | {erro:.6f}")
if historico_reserva:
    epoca, erro = historico_reserva[-1]
    print(f"{epoca:>5} | {erro:.6f}")

Arquitetura: 4 entradas -> 12 neurônios ocultos -> 1 saída
Pesos ajustáveis, incluindo os vieses: 73
Entradas: SYSTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES
Configuração: ('Quatro medidas', 'desvio', 'mediana', 12, 4.0, 4000)
Época | Erro quadrático de treinamento
  100 | 0.084841
  500 | 0.080549
  900 | 0.070417
 1300 | 0.064569
 1700 | 0.054375
 2100 | 0.042685
 2500 | 0.035984
 2900 | 0.032120
 3300 | 0.029874
 3700 | 0.028611
 4000 | 0.027946


### Registro da busca

O registro contém a configuração, a matriz de confusão por rodada, as conferências e a avaliação local. A ordem das contagens é TN, FP, FN e TP, com diabetes como classe 1.

As 104 amostras já foram observadas antes neste projeto. As conferências usam as mesmas pessoas em divisões ou inicializações diferentes.

In [9]:
caminho_busca = raiz / "resultados" / "rede_neural_aulas_busca.npz"
np.savez_compressed(
    caminho_busca,
    catalogo=json.dumps({
        "configuracoes": configuracoes_completas, "grupos": grupos,
        "semente_divisao_escolha": 42, "semente_pesos_escolha": 7,
        "conferencias": list(conferencias),
        "sha256_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
        "atualizacao": "gradiente médio em lote por época",
    }),
    busca=contagens_completas, escolhida=escolhida,
    conferencias=np.array(list(conferencias.values())), reserva=reserva,
)
print("Registro:", caminho_busca.relative_to(raiz))

Registro: resultados\rede_neural_aulas_busca.npz


### Treinamento final e arquivo de submissão

Fixamos a configuração e treinamos novamente com todas as 520 amostras rotuladas. A preparação do teste repete a padronização da exploração. O teste só recebe as transformações aprendidas no treinamento e as previsões da rede pronta.

Salvamos o modelo, os parâmetros de preparação e o CSV de previsões. Isso permite conferir as previsões após reabrir os arquivos, sem treinar outra rede.

In [10]:
def padronizar_teste(amostra):
    entrada = amostra.copy()
    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""
    altura = ler_numero(entrada, "HEIGHT")
    if np.isfinite(altura) and altura < 3:
        entrada["HEIGHT"] = str(altura * 100)
    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"], entrada["DIASTOLIC"] = "", ""
    else:
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
        entrada["DIASTOLIC"] = str(int(diastolica))

    genero = entrada["GENDER"].strip().upper()
    entrada["GENDER"] = {"MALE": "M", "M": "M", "FEMALE": "F", "F": "F", "": "N"}.get(genero, genero)
    mapas = {
        "SMOKER": {"never": "nunca_fumou", "former": "ex_fumante", "ex-smoker": "ex_fumante", "current": "fumante_atual", "yes": "fumante_atual", "": ""},
        "PHYSICAL_ACTIVITY": {"sedentary": "sedentario", "sedentry": "sedentario", "light": "leve", "ligth": "leve", "moderate": "moderada", "moderat": "moderada", "intense": "intensa", "intensive": "intensa", "": ""},
        "FAMILY_HISTORY": {"yes": "sim", "y": "sim", "sim": "sim", "1": "sim", "no": "nao", "n": "nao", "não": "nao", "nao": "nao", "0": "nao", "unknown": "desconhecido", "desconhecido": "desconhecido", "": "desconhecido"},
    }
    for coluna, mapa in mapas.items():
        valor = entrada[coluna].strip().lower()
        entrada[coluna] = mapa.get(valor, valor)
    sangue = entrada["BLOOD_TYPE"].strip()
    entrada["BLOOD_GROUP"] = sangue[:-1] if sangue else ""
    entrada["RH_FACTOR"] = sangue[-1] if sangue else ""
    return entrada


with (raiz / "dados" / "kaggle-test" / "test.csv").open(encoding="utf-8-sig", newline="") as arquivo:
    teste_original = list(csv.DictReader(arquivo))
ids_teste = [a["id"] for a in teste_original]
if not ids_teste or len(set(ids_teste)) != len(ids_teste) or any(not i.strip() for i in ids_teste):
    raise ValueError("O teste precisa ter IDs únicos e preenchidos.")
teste = [padronizar_teste(a) for a in teste_original]

grupo, escala, preenchimento, ocultos, taxa, epoca = configuracao_final
preparo_final = ajustar_preparo(amostras, grupo, escala, preenchimento)
x_final = transformar(amostras, preparo_final)
x_teste = transformar(teste, preparo_final)
modelos_finais, historico_final = treinar(
    x_final, classes, ocultos, taxa, (epoca,), semente=7, registrar=True
)
modelo_final = modelos_finais[epoca]
previsoes_teste = prever(x_teste, modelo_final)

caminho_modelo = raiz / "resultados" / "rede_neural_aulas_modelo.npz"
np.savez_compressed(
    caminho_modelo, pesos_entrada=modelo_final[0], pesos_saida=modelo_final[1],
    preparo=json.dumps(preparo_final, ensure_ascii=False),
    configuracao=json.dumps(configuracao_final, ensure_ascii=False),
    semente_pesos=7,
    sha256_base=hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
)
caminho_envio = raiz / "resultados" / "submissao_rede_neural_aulas.csv"
with caminho_envio.open("w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["id", "class"])
    escritor.writerows(zip(ids_teste, previsoes_teste.tolist(), strict=True))

with np.load(caminho_modelo, allow_pickle=False) as salvo:
    preparo_carregado = json.loads(salvo["preparo"].item())
    modelo_carregado = (salvo["pesos_entrada"], salvo["pesos_saida"])
    previsoes_carregadas = prever(transformar(teste, preparo_carregado), modelo_carregado)
if not np.array_equal(previsoes_teste, previsoes_carregadas):
    raise ValueError("O modelo salvo não reproduziu as previsões.")

print("Modelo treinado com:", len(amostras), "amostras rotuladas")
print("Modelo salvo:", caminho_modelo.relative_to(raiz))
print("CSV salvo:", caminho_envio.relative_to(raiz))
print("Previsões:", len(previsoes_teste))
print("SHA-256 do CSV:", hashlib.sha256(caminho_envio.read_bytes()).hexdigest())

Modelo treinado com: 520 amostras rotuladas
Modelo salvo: resultados\rede_neural_aulas_modelo.npz
CSV salvo: resultados\submissao_rede_neural_aulas.csv
Previsões: 312
SHA-256 do CSV: d7dfe60a5ce289ba2910569c9efc4a026243ae3fe01d2923c92581af92a18781


### Referências

- [Aula 6](../docs/materiais_aula/aula6_MD.pdf), arquitetura, sigmoide, gradiente e retropropagação.
- [Aula 2](../docs/materiais_aula/aula2_MD.pdf), dados, seleção de atributos e binarização.
- [Aula 4](../docs/materiais_aula/aula4_MD.pdf), média e mediana.
- [Aula 5](../docs/materiais_aula/aula5_MD.pdf), normalização pelo mínimo e máximo.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf), avaliação e métricas.
- [Aula 10](../docs/materiais_aula/aula10_MD.pdf), normalização pela média e desvio padrão.

A atualização em lote, a grade de configurações e as sementes são escolhas desta implementação. O limite de classificação permanece em 0,5.